# Classer des intentions avec des encodeurs

Des demandes à un assistant vocal (« réveille-moi à neuf heures », « mets du jazz »), chacune avec l'une de 60 intentions : un vrai problème de classification, avec des classes nombreuses, déséquilibrées et parfois proches. Les données : [MASSIVE](https://github.com/alexa/massive), d'Amazon.

1. Les modèles classiques sur TF-IDF : la référence, puis des n-grammes de caractères
2. *Embeddings* figés + régression logistique
3. Affiner un encodeur
4. Garder les résultats pour demain (le TP suivant compare avec un LLM), puis lire le F1 de près
5. Bonus : SetFit avec 8 exemples par classe, et la calibration

Ce notebook va plus vite avec un GPU : *Exécution > Modifier le type d'exécution > GPU T4*. Sans GPU, il tourne quand même : les modèles des exercices 2, 3 et 5, trop lents pour un processeur, n'y voient qu'un échantillon, puis l'exercice suivant charge leur résultat calculé sur GPU.

## Données

In [ ]:
!wget -q https://github.com/shuuchuu/datasets/raw/refs/heads/main/ghsparse.sh -O /tmp/ghsparse.sh && bash /tmp/ghsparse.sh massive-intents

In [ ]:
import time
from collections import Counter

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import accuracy_score, f1_score

train = pd.read_parquet("massive-intents/fr-train.parquet")
test = pd.read_parquet("massive-intents/fr-test.parquet")
intentions = sorted(train.label.unique())
GPU = torch.cuda.is_available()
DEVICE = "cuda" if GPU else "cpu"
# Sur CPU, float32 : les demi-précisions (float16, et bfloat16, celle des poids publiés,
# qu'on aurait par défaut) y sont très lentes.
DTYPE = torch.float16 if GPU else torch.float32
print("GPU" if GPU else "Pas de GPU : les calculs longs chargeront leur résultat")
# Ce que voient les modèles des exercices 2, 3 et 5 : tout sur GPU ; sans GPU, les 64
# premiers textes d'entraînement et de test.
n = None if GPU else 64
train_part, test_part = train[:n], test[:n]
print(
  len(train),
  "exemples d'entraînement,",
  len(test),
  "de test,",
  len(intentions),
  "intentions",
)
train.label.value_counts().describe()

`enregistre` mesure et garde les résultats de chaque méthode : le F1 macro (chaque intention du test compte autant), l'*accuracy*, le temps d'entraînement et le temps de prédiction par texte. `confusions` affiche les paires d'intentions les plus souvent confondues.

In [ ]:
resultats = {}


def enregistre(
  nom: str, predictions, secondes_ent: float, secondes_pred: float
) -> None:
  # Sans GPU, une méthode peut n'avoir prédit que les premiers textes du test.
  vrai = test.label[: len(predictions)]
  # labels : le F1 macro porte sur les seules intentions du test. Par défaut, une
  # intention absente du test (cooking_query) mais prédite une fois y compterait,
  # avec un F1 de 0.
  resultats[nom] = {
    "predictions": list(predictions),
    "entrainement_s": secondes_ent,
    "ms_par_texte": 1000 * secondes_pred / len(predictions),
  }
  print(
    f"{nom:30} F1 macro {f1_score(vrai, predictions, average='macro', labels=vrai.unique()):.3f}"
    f"  accuracy {accuracy_score(vrai, predictions):.3f}"
    f"  entraînement {secondes_ent:5.0f} s  {resultats[nom]['ms_par_texte']:.2f} ms/texte"
  )


def confusions(predictions, n: int = 10) -> list[tuple[tuple[str, str], int]]:
  erreurs = Counter((t, p) for t, p in zip(test.label, predictions) if t != p)
  return erreurs.most_common(n)

## 1. La référence : les modèles classiques sur TF-IDF

Les modèles du programme officiel, sur une représentation TF-IDF (mots et paires de mots). Lancez la cellule (deux à trois minutes) et lisez les résultats : lequel est le meilleur ? Quelles intentions confond-il ?

In [ ]:
from lightgbm import LGBMClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC

vectoriseur = TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True, min_df=2)
X_train = vectoriseur.fit_transform(train.text)
X_test = vectoriseur.transform(test.text)
for nom, modele in (
  ("TF-IDF + Naive Bayes", MultinomialNB(alpha=0.1)),
  ("TF-IDF + SVM linéaire", LinearSVC()),
  ("TF-IDF + régression logistique", LogisticRegression(max_iter=2000, C=10)),
  (
    "TF-IDF + forêt aléatoire",
    RandomForestClassifier(n_estimators=200, n_jobs=-1, random_state=0),
  ),
  (
    "TF-IDF + LightGBM",
    LGBMClassifier(learning_rate=0.05, num_leaves=15, verbose=-1),
  ),
):
  debut = time.time()
  modele.fit(X_train, train.label)
  duree_ent = time.time() - debut
  debut = time.time()
  predictions = modele.predict(X_test)
  enregistre(nom, predictions, duree_ent, time.time() - debut)

# Les arbres (forêts, gradient boosting), rois des données tabulaires, s'en sortent mal
# sur des milliers de colonnes creuses. LightGBM (gradient boosting) diverge même avec
# ses réglages par défaut (F1 proche de 0) ; avec un pas et des arbres plus petits
# (learning_rate=0.05, num_leaves=15), il apprend, mais reste sous la forêt. Sur du
# texte, la référence classique est un modèle linéaire sur TF-IDF.

In [ ]:
confusions(resultats["TF-IDF + SVM linéaire"]["predictions"])

### Des n-grammes de caractères

Ces demandes sont courtes, orales, et pas toujours bien orthographiées (« quelles sont les actualite », « je veux que la musique soit enregistrer »). Pour le SVM linéaire, remplacez les mots par des n-grammes de 2 à 5 caractères, pris à l'intérieur des mots, les autres réglages de `TfidfVectorizer` inchangés, et enregistrez le résultat avec `enregistre`. Comparez au SVM sur les mots : que gagne-t-on, et pourquoi sur ces textes ?

**Documentation utile**

- [`TfidfVectorizer`](https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html)

In [ ]:
# Votre code ici

#### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- `TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 5), sublinear_tf=True, min_df=2)` : `"char_wb"` découpe chaque mot, entouré d'une espace de chaque côté, en n-grammes de caractères ; `ngram_range=(2, 5)` : de 2 à 5 caractères
- `.fit_transform(train.text)` sur l'entraînement, `.transform(test.text)` sur le test, comme pour les mots
- `LinearSVC().fit(X, train.label)`, puis `.predict(X_test)`, chronométrés comme dans la boucle

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
vectoriseur_carac = TfidfVectorizer(
  analyzer=..., ngram_range=..., sublinear_tf=True, min_df=2
)
X_train_carac = vectoriseur_carac.fit_transform(train.text)
X_test_carac = vectoriseur_carac.transform(...)
debut = time.time()
svm_carac = LinearSVC().fit(..., train.label)
duree_ent = time.time() - debut
debut = time.time()
predictions = svm_carac.predict(...)
enregistre("TF-IDF caractères + SVM", predictions, duree_ent, time.time() - debut)
```

#### Solution

In [ ]:
vectoriseur_carac = TfidfVectorizer(
  analyzer="char_wb", ngram_range=(2, 5), sublinear_tf=True, min_df=2
)
X_train_carac = vectoriseur_carac.fit_transform(train.text)
X_test_carac = vectoriseur_carac.transform(test.text)
debut = time.time()
svm_carac = LinearSVC().fit(X_train_carac, train.label)
duree_ent = time.time() - debut
debut = time.time()
predictions = svm_carac.predict(X_test_carac)
enregistre("TF-IDF caractères + SVM", predictions, duree_ent, time.time() - debut)

# Sur les mots, F1 macro 0,777 et accuracy 0,823 ; sur les caractères, 0,796 et 0,839.
# Un mot jamais vu à l'entraînement a souvent des morceaux connus : « tamisées » n'y
# est pas, mais « tamisée » et « tamiser » y sont. Les n-grammes de caractères
# rapprochent ainsi les formes d'un même mot, et résistent aux fautes et aux accents
# oubliés (« actualite »). Dans un texte de quelques mots, chaque mot reconnu compte.

## 2. *Embeddings* figés + régression logistique

Encodez les textes d'entraînement et de test avec Qwen3-Embedding-0.6B (`embedder`), en précédant chaque texte de l'instruction ci-dessous, qui décrit la tâche au modèle. Entraînez une régression logistique sur les vecteurs, et enregistrez le résultat avec `enregistre` (le temps de prédiction comprend l'encodage des textes de test). Travaillez sur `train_part` et `test_part` : tout le jeu sur GPU ; sans GPU, 64 textes de chaque (une minute environ), et l'exercice suivant charge les vecteurs de tous, calculés sur GPU.

**Documentation utile**

- [`SentenceTransformer.encode`](https://sbert.net/docs/package_reference/sentence_transformer/model.html#sentence_transformers.sentence_transformer.model.SentenceTransformer.encode)

In [ ]:
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer(
  "Qwen/Qwen3-Embedding-0.6B", device=DEVICE, model_kwargs={"dtype": DTYPE}
)
embedder.max_seq_length = 64
INSTRUCTION = "Instruct: Classify the intent of a request to a voice assistant\nQuery: "

In [ ]:
# Votre code ici

### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- `embedder.encode(textes, prompt=INSTRUCTION, batch_size=64)`, sur `train_part.text.tolist()` puis `test_part.text.tolist()`
- `LogisticRegression(max_iter=2000, C=10).fit(vecteurs, train_part.label)`, puis `.predict(vecteurs_test)`
- `enregistre(nom, predictions, durée_d_entraînement, durée_de_prédiction)` : mesurez avec `time.time()` ; l'entraînement comprend l'encodage des textes d'entraînement, la prédiction l'encodage des textes de test

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
debut = time.time()
E_train = embedder.encode(..., prompt=INSTRUCTION, batch_size=64)
classifieur = LogisticRegression(max_iter=2000, C=10).fit(..., ...)
duree_ent = time.time() - debut
debut = time.time()
E_test = embedder.encode(..., prompt=INSTRUCTION, batch_size=64)
predictions = classifieur.predict(...)
enregistre("Embeddings + régression", predictions, duree_ent, time.time() - debut)
```

### Solution

In [ ]:
debut = time.time()
E_train = embedder.encode(train_part.text.tolist(), prompt=INSTRUCTION, batch_size=64)
classifieur = LogisticRegression(max_iter=2000, C=10).fit(E_train, train_part.label)
duree_ent = time.time() - debut
debut = time.time()
E_test = embedder.encode(test_part.text.tolist(), prompt=INSTRUCTION, batch_size=64)
predictions = classifieur.predict(E_test)
enregistre("Embeddings + régression", predictions, duree_ent, time.time() - debut)

## 3. Affiner un encodeur

Affinez `ENCODEUR` (un encodeur pré-entraîné, plus une couche de classification) sur les données d'entraînement, avec `transformers` : tokenisez les textes, numérotez les intentions, et entraînez avec un `Trainer`.

L'entraînement prend quelques minutes : lancez-le, et pendant ce temps relisez les résultats des étapes 1 et 2. Prédisez ensuite les intentions du test (`trainer.predict`, sur `test_part`) et enregistrez le résultat.

Sans GPU, l'entraînement complet prendrait des heures : `NB_PAS` l'arrête après 10 pas, pour voir la boucle tourner, et la prédiction ne porte que sur 64 textes. L'exercice suivant charge alors les logits du test du modèle entraîné sur GPU, ce qu'il faut à la suite du TP (le modèle lui-même, 600 Mo par langue, n'est pas publié). Leur F1 macro est de 0,835 ; celui d'un run sur GPU peut s'en écarter de quelques points (0,83 à 0,86 dans les nôtres) : les calculs sur GPU ne se reproduisent pas exactement d'une carte ou d'une version des bibliothèques à l'autre.

**Documentation utile**

- [Affiner un modèle pré-entraîné](https://huggingface.co/docs/transformers/training)
- [`Dataset.from_dict`](https://huggingface.co/docs/datasets/package_reference/main_classes#datasets.Dataset.from_dict)
- [`Dataset.map`](https://huggingface.co/docs/datasets/package_reference/main_classes#datasets.Dataset.map)
- [`tokenizer(...)`](https://huggingface.co/docs/transformers/main_classes/tokenizer#transformers.TokenizersBackend.__call__)
- [`AutoModelForSequenceClassification`](https://huggingface.co/docs/transformers/model_doc/auto#transformers.AutoModelForSequenceClassification)
- [`TrainingArguments`](https://huggingface.co/docs/transformers/main_classes/trainer#transformers.TrainingArguments)
- [`Trainer`](https://huggingface.co/docs/transformers/main_classes/trainer#transformers.Trainer)
- [`DataCollatorWithPadding`](https://huggingface.co/docs/transformers/main_classes/data_collator#transformers.DataCollatorWithPadding)
- [`Trainer.predict`](https://huggingface.co/docs/transformers/main_classes/trainer#transformers.Trainer.predict)

In [ ]:
# Sans GPU, l'exercice précédent n'a encodé que 64 textes de chaque : on charge les vecteurs
# de tous, calculés sur GPU par le même code (stockés en float16), on refait la régression,
# et on enregistre son résultat avec les temps mesurés sur GPU.
if not GPU:
  E_train = np.load("massive-intents/lab-results/embeddings-train-fr.npy").astype(
    np.float32
  )
  E_test = np.load("massive-intents/lab-results/embeddings-test-fr.npy").astype(
    np.float32
  )
  classifieur = LogisticRegression(max_iter=2000, C=10).fit(E_train, train.label)
  gpu = pd.read_csv("massive-intents/lab-results/timings-fr.csv").set_index("method")
  nom = "Embeddings + régression"
  secondes_pred = gpu.ms_per_text[nom] * len(test) / 1000
  enregistre(nom, classifieur.predict(E_test), gpu.train_s[nom], secondes_pred)

In [ ]:
# Multilingue, 2025 ; le meilleur des encodeurs testés sur ces données
ENCODEUR = "jhu-clsp/mmBERT-base"
# Pour CamemBERTav2, AutoTokenizer (transformers 5) découpe le texte par caractères :
# 0,3 de F1 macro environ. Avec le tokenizer de son tokenizer.json, 0,68 à 0,72.
# 3 époques sur GPU (quelques minutes) ; sans GPU, 10 pas (deux minutes environ).
NB_PAS = -1 if GPU else 10

In [ ]:
# Votre code ici

### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- numéroter les intentions : `index = {label: i for i, label in enumerate(intentions)}`
- `tokenizer(lot["text"], truncation=True, max_length=64)`, appliqué par `Dataset.map(..., batched=True, remove_columns=["text"])` : la fonction reçoit des lots de textes, et le `Trainer` ne veut que des colonnes numériques
- `Dataset.from_dict({"text": ..., "label": ...})` : un jeu de données à partir de listes
- `AutoModelForSequenceClassification.from_pretrained(ENCODEUR, num_labels=len(intentions))`
- `TrainingArguments("encodeur", num_train_epochs=3, learning_rate=5e-5, per_device_train_batch_size=32, fp16=GPU, max_steps=NB_PAS, warmup_steps=0.1, save_strategy="no", report_to="none")` ; `fp16=GPU` : la demi-précision, sur GPU seulement ; `max_steps=-1` : pas de limite, les 3 époques ; `warmup_steps=0.1` : un nombre inférieur à 1 est une proportion des pas d'entraînement
- `Trainer(model=..., args=..., train_dataset=..., data_collator=DataCollatorWithPadding(tokenizer))` : le *collator* complète chaque lot à la longueur de son plus long texte
- `trainer.predict(en_dataset(test_part)).predictions` : les logits, une ligne par texte ; `argmax(axis=1)` donne la classe prédite, à convertir en intention avec `intentions[i]`

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
from datasets import Dataset
from transformers import (
  AutoModelForSequenceClassification,
  AutoTokenizer,
  DataCollatorWithPadding,
  Trainer,
  TrainingArguments,
)

index = {label: i for i, label in enumerate(intentions)}
tokenizer = AutoTokenizer.from_pretrained(ENCODEUR)


def en_dataset(df: pd.DataFrame) -> Dataset:
  jeu = Dataset.from_dict(
    {"text": df.text.tolist(), "label": [... for label in df.label]}
  )  # le numéro de chaque intention
  return jeu.map(
    lambda lot: tokenizer(lot["text"], truncation=True, max_length=64),
    batched=True,
    remove_columns=[...],
  )


modele = AutoModelForSequenceClassification.from_pretrained(ENCODEUR, num_labels=...)
args = TrainingArguments(
  "encodeur",
  num_train_epochs=3,
  learning_rate=5e-5,
  per_device_train_batch_size=32,
  per_device_eval_batch_size=128,
  fp16=GPU,
  max_steps=NB_PAS,
  warmup_steps=0.1,
  logging_steps=100,
  save_strategy="no",
  report_to="none",
)
trainer = Trainer(
  model=modele,
  args=args,
  train_dataset=...,
  data_collator=DataCollatorWithPadding(tokenizer),
)
debut = time.time()
trainer.train()
duree_ent = time.time() - debut

debut = time.time()
logits = trainer.predict(...).predictions
predictions = [intentions[i] for i in ...]  # la classe prédite de chaque texte
enregistre("Encodeur affiné", predictions, duree_ent, time.time() - debut)
```

### Solution

In [ ]:
from datasets import Dataset
from transformers import (
  AutoModelForSequenceClassification,
  AutoTokenizer,
  DataCollatorWithPadding,
  Trainer,
  TrainingArguments,
)

index = {label: i for i, label in enumerate(intentions)}
tokenizer = AutoTokenizer.from_pretrained(ENCODEUR)


def en_dataset(df: pd.DataFrame) -> Dataset:
  jeu = Dataset.from_dict(
    {"text": df.text.tolist(), "label": [index[label] for label in df.label]}
  )
  return jeu.map(
    lambda lot: tokenizer(lot["text"], truncation=True, max_length=64),
    batched=True,
    remove_columns=["text"],
  )


modele = AutoModelForSequenceClassification.from_pretrained(
  ENCODEUR, num_labels=len(intentions)
)
args = TrainingArguments(
  "encodeur",
  num_train_epochs=3,
  learning_rate=5e-5,
  per_device_train_batch_size=32,
  per_device_eval_batch_size=128,
  fp16=GPU,
  max_steps=NB_PAS,
  warmup_steps=0.1,
  logging_steps=100,
  save_strategy="no",
  report_to="none",
)
trainer = Trainer(
  model=modele,
  args=args,
  train_dataset=en_dataset(train),
  data_collator=DataCollatorWithPadding(tokenizer),
)
debut = time.time()
trainer.train()
duree_ent = time.time() - debut

debut = time.time()
logits = trainer.predict(en_dataset(test_part)).predictions
predictions = [intentions[i] for i in logits.argmax(axis=1)]
enregistre("Encodeur affiné", predictions, duree_ent, time.time() - debut)

## 4. Garder les résultats, et lire le F1 de près

Demain, le TP suivant compare ces méthodes à un LLM. La cellule suivante enregistre les prédictions de chaque méthode dans `predictions-fr.csv` (une ligne par texte de test : `id`, `label`, puis une colonne par méthode) et les temps dans `timings-fr.csv`. Lancez-la, puis téléchargez les deux fichiers (panneau *Fichiers* à gauche, ou `google.colab.files.download`) : le TP suivant les charge. Une copie de nos résultats est publiée, au cas où.

In [ ]:
# Sans GPU, l'encodeur n'a fait que 10 pas d'entraînement : on charge les logits du test du
# modèle entraîné sur GPU par le même code, et on enregistre son résultat avec les temps
# mesurés sur GPU.
if not GPU:
  logits = np.load("massive-intents/lab-results/encoder-logits-fr.npy")
  gpu = pd.read_csv("massive-intents/lab-results/timings-fr.csv").set_index("method")
  nom = "Encodeur affiné"
  predictions = [intentions[i] for i in logits.argmax(axis=1)]
  secondes_pred = gpu.ms_per_text[nom] * len(test) / 1000
  enregistre(nom, predictions, gpu.train_s[nom], secondes_pred)

# Les prédictions et les temps de chaque méthode, pour le TP suivant
predictions = pd.DataFrame({"id": test.id, "label": test.label})
for nom, resultat in resultats.items():
  predictions[nom] = resultat["predictions"]
predictions.to_csv("predictions-fr.csv", index=False)
# Les colonnes de nos fichiers publiés, que lit le TP suivant
durees = pd.DataFrame(
  [(nom, r["entrainement_s"], r["ms_par_texte"]) for nom, r in resultats.items()],
  columns=["method", "train_s", "ms_per_text"],
)
durees.to_csv("timings-fr.csv", index=False)
durees

### Ce que cache le F1 macro

Un F1 macro résume 60 intentions en un chiffre. Avec `classification_report`, listez les 5 intentions au plus faible F1 de l'encodeur affiné, et rapprochez-les de leur nombre de textes de test (le *support*) et de leurs confusions (`confusions`) : lesquelles sont de vraies faiblesses ? Comptez ensuite les textes de test déjà vus à l'entraînement (en minuscules, sans espaces en début ni en fin), et recalculez sans eux le F1 macro de chaque méthode de `resultats` : que mesurait-on ?

**Documentation utile**

- [`classification_report`](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.classification_report.html)
- [`Series.isin`](https://pandas.pydata.org/docs/reference/api/pandas.Series.isin.html)

In [ ]:
# Votre code ici

#### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- `encodeur = resultats["Encodeur affiné"]["predictions"]` : les prédictions de l'encodeur, une par texte de test
- `classification_report(test.label, encodeur, output_dict=True, zero_division=0)` : un dictionnaire, une entrée par intention (`precision`, `recall`, `f1-score`, `support`) plus `accuracy` et deux moyennes ; `zero_division=0` compte 0, sans avertissement, pour une intention jamais prédite ou sans texte de test
- `pd.DataFrame(rapport).T` : une ligne par intention ; `.drop(["accuracy", "macro avg", "weighted avg"])` retire les autres, `.sort_values("f1-score").head(5)` garde les 5 plus faibles
- `confusions(encodeur, n=None)` : toutes les paires (vraie, prédite) confondues, de la plus fréquente à la plus rare ; gardez celles où l'intention apparaît
- `test.text.str.lower().str.strip()` : les textes en minuscules, sans espaces en bout ; `.isin(set(...))` dit, pour chaque texte de test, s'il est dans l'entraînement
- `f1_score(vrai, predictions[~vus], average="macro", labels=vrai.unique())`, avec `vrai = test.label[~vus]`, et `vus` et `predictions` en tableaux `numpy` : le F1 macro sans les textes vus (`~` inverse un masque de booléens), sur les intentions de ces textes, comme dans `enregistre`

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
from sklearn.metrics import classification_report

encodeur = resultats["Encodeur affiné"]["predictions"]
rapport = classification_report(test.label, encodeur, output_dict=True, zero_division=0)
par_classe = pd.DataFrame(rapport).T.drop(["accuracy", "macro avg", "weighted avg"])
plus_faibles = par_classe.sort_values(...).head(5)
print(plus_faibles.round(2))
for intention in plus_faibles.index:
  paires = [(paire, nb) for paire, nb in confusions(encodeur, n=None) if ...]
  print(intention, paires[:3])  # ses trois confusions les plus fréquentes


def normaliser(textes: pd.Series) -> pd.Series:
  return ...  # en minuscules, sans espaces en bout


vus = normaliser(test.text).isin(set(normaliser(...))).to_numpy()
print(vus.sum(), "textes de test sur", len(test), "déjà vus à l'entraînement")
for nom, resultat in resultats.items():
  predictions = np.array(resultat["predictions"])
  vrai = test.label[...]
  f1 = f1_score(vrai, predictions[...], average="macro", labels=vrai.unique())
  print(f"{nom:30} F1 macro sans eux {f1:.3f}")
```

#### Solution

In [ ]:
from sklearn.metrics import classification_report

encodeur = resultats["Encodeur affiné"]["predictions"]
rapport = classification_report(test.label, encodeur, output_dict=True, zero_division=0)
par_classe = pd.DataFrame(rapport).T.drop(["accuracy", "macro avg", "weighted avg"])
plus_faibles = par_classe.sort_values("f1-score").head(5)
print(plus_faibles.round(2))
for intention in plus_faibles.index:
  paires = [
    (paire, nb) for paire, nb in confusions(encodeur, n=None) if intention in paire
  ]
  print(intention, paires[:3])


def normaliser(textes: pd.Series) -> pd.Series:
  return textes.str.lower().str.strip()


vus = normaliser(test.text).isin(set(normaliser(train.text))).to_numpy()
print(vus.sum(), "textes de test sur", len(test), "déjà vus à l'entraînement")
for nom, resultat in resultats.items():
  predictions = np.array(resultat["predictions"])
  vrai = test.label[~vus]
  f1 = f1_score(vrai, predictions[~vus], average="macro", labels=vrai.unique())
  print(f"{nom:30} F1 macro sans eux {f1:.3f}")

# Avec nos logits publiés (sans GPU) ; ceux d'un encodeur entraîné sur GPU donnent des
# chiffres voisins. Les plus faibles : cooking_query, sans aucun texte de test (une
# seule prédiction, fausse, lui donne un F1 de 0, compté dans la moyenne de
# classification_report, pas dans celle d'enregistre),
# general_greet et iot_hue_lighton (1 et 3 textes : un F1 sur si peu ne dit presque
# rien), music_settings (6 textes, confondu avec play_music) et general_quirky (169
# textes, F1 0,66). Seule general_quirky est une vraie faiblesse : le fourre-tout des
# demandes sans intention précise, confondu avec qa_factoid et news_query.
# 150 textes de test sur 2 974 sont déjà dans l'entraînement (« silence », « quelle
# est la date ») : l'encodeur en classe bien 99 %, contre 87 % des autres. Sans eux,
# son F1 passe de 0,835 à 0,828, celui de la régression sur TF-IDF de 0,773 à
# 0,762 ; chaque méthode perd d'un demi-point à près de deux points, sans que leur
# classement change. Le test mesurait un peu de mémoire en plus de la généralisation.

## 5. Bonus

### SetFit

Avec seulement 8 exemples par intention, affinez un modèle d'*embeddings* par contraste puis une tête de classification ([SetFit](https://github.com/huggingface/setfit)). Où se place-t-il, avec environ 24 fois moins d'étiquettes ? Sans GPU, `max_steps=NB_PAS` arrête l'apprentissage par contraste après 10 pas, et la prédiction porte sur `test_part` : quelques minutes au lieu de trois heures environ, pour un résultat moins bon ; le F1 de l'apprentissage complet, sur GPU, est dans la solution.

**Documentation utile**

- [SetFit : démarrage rapide](https://huggingface.co/docs/setfit/quickstart)
- [`DataFrame.sample`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.sample.html)
- [`DataFrameGroupBy.head`](https://pandas.pydata.org/docs/reference/api/pandas.api.typing.DataFrameGroupBy.head.html)
- [`setfit.TrainingArguments`](https://huggingface.co/docs/setfit/reference/trainer#setfit.TrainingArguments)
- [`multilingual-e5-base`](https://huggingface.co/intfloat/multilingual-e5-base)

In [ ]:
# Votre code ici

#### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- `train.sample(frac=1, random_state=0).groupby("label").head(8)` : mélange les exemples, puis garde les 8 premiers de chaque intention (l'une d'elles n'en a que 4)
- `SetFitModel.from_pretrained("intfloat/multilingual-e5-base", labels=intentions)` : un bon modèle de départ, dont les textes doivent commencer par `"query: "`
- `setfit.TrainingArguments(batch_size=32, num_epochs=1, num_iterations=10, max_steps=NB_PAS, report_to="none")` : `num_iterations` est le nombre de paires générées par exemple pour l'apprentissage par contraste ; `max_steps=NB_PAS` : sans limite sur GPU, 10 pas sans GPU (l'exercice 3)
- `setfit.Trainer(model=..., args=..., train_dataset=Dataset.from_dict({"text": ..., "label": ...}))`, puis `.train()` ; `modèle.predict(liste_de_textes)` pour prédire
- `%pip install -q setfit` en première ligne de la cellule ; importez `Trainer` et `TrainingArguments` de `setfit` sous d'autres noms (`SetFitTrainer`, `SetFitArguments`) pour ne pas écraser ceux de `transformers`

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
%pip install -q setfit
from datasets import Dataset as HFDataset
from setfit import (
  SetFitModel,
  Trainer as SetFitTrainer,
  TrainingArguments as SetFitArguments,
)

quelques = train.sample(frac=1, random_state=0).groupby("label").head(...)
modele_setfit = SetFitModel.from_pretrained(
  "intfloat/multilingual-e5-base", labels=intentions
)
trainer_setfit = SetFitTrainer(
  model=modele_setfit,
  args=SetFitArguments(
    batch_size=32,
    num_epochs=1,
    num_iterations=10,
    max_steps=NB_PAS,
    report_to="none",
  ),
  train_dataset=HFDataset.from_dict(
    {"text": ("query: " + quelques.text).tolist(), "label": ...}
  ),
)
debut = time.time()
trainer_setfit.train()
duree_ent = time.time() - debut
debut = time.time()
predictions = modele_setfit.predict(...)  # les textes de test_part, précédés de "query: "
enregistre("SetFit (8 par classe)", list(predictions), duree_ent, time.time() - debut)
```

#### Solution

In [ ]:
%pip install -q setfit
from datasets import Dataset as HFDataset
from setfit import (
  SetFitModel,
  Trainer as SetFitTrainer,
  TrainingArguments as SetFitArguments,
)

quelques = train.sample(frac=1, random_state=0).groupby("label").head(8)
modele_setfit = SetFitModel.from_pretrained(
  "intfloat/multilingual-e5-base", labels=intentions
)
trainer_setfit = SetFitTrainer(
  model=modele_setfit,
  args=SetFitArguments(
    batch_size=32,
    num_epochs=1,
    num_iterations=10,
    max_steps=NB_PAS,
    report_to="none",
  ),
  train_dataset=HFDataset.from_dict(
    {"text": ("query: " + quelques.text).tolist(), "label": quelques.label.tolist()}
  ),
)
debut = time.time()
trainer_setfit.train()
duree_ent = time.time() - debut
debut = time.time()
predictions = modele_setfit.predict(("query: " + test_part.text).tolist())
enregistre("SetFit (8 par classe)", list(predictions), duree_ent, time.time() - debut)

# Sur GPU, une minute d'entraînement environ, et un F1 macro de 0,617 sur tout le test :
# avec 8 exemples par intention, SetFit reste loin de l'encodeur affiné sur tout
# l'entraînement (0,83 à 0,86), et même sous TF-IDF + Naive Bayes (0,725). Sans GPU,
# 10 pas et 64 textes de test ne disent presque rien de ce qu'il vaut.

### Calibration

Pour la régression sur *embeddings* et l'encodeur affiné, tracez le diagramme de fiabilité : la part de prédictions justes en fonction de la probabilité annoncée (`sklearn.calibration.calibration_curve`, sur « la prédiction est-elle juste ? » et la probabilité de la classe prédite). Lequel est le mieux calibré ?

**Documentation utile**

- [`calibration_curve`](https://scikit-learn.org/stable/modules/generated/sklearn.calibration.calibration_curve.html)
- [`LogisticRegression.predict_proba`](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html#sklearn.linear_model.LogisticRegression.predict_proba)
- [`scipy.special.softmax`](https://docs.scipy.org/doc/scipy/reference/generated/scipy.special.softmax.html)

In [ ]:
# Votre code ici

#### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- `calibration_curve(justes, confiance, n_bins=10, strategy="quantile")` : `justes` dit si chaque prédiction est juste (booléens), `confiance` est la probabilité annoncée ; `strategy="quantile"` met autant de prédictions dans chaque intervalle ; renvoie la part de justes et la confiance moyenne de chaque intervalle
- `classifieur.predict_proba(E_test)` : une ligne par texte, les colonnes suivent l'ordre de `classifieur.classes_`
- `softmax(logits, axis=1)` : les logits de l'encodeur affiné, transformés en probabilités ; ses colonnes suivent l'ordre de `intentions`
- `proba.max(axis=1)` : la probabilité de la classe prédite ; `proba.argmax(axis=1)` : son numéro

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
import matplotlib.pyplot as plt
from scipy.special import softmax
from sklearn.calibration import calibration_curve

probabilites = {
  "Embeddings + régression": classifieur.predict_proba(E_test),
  "Encodeur affiné": softmax(..., axis=1),
}
for nom, proba in probabilites.items():
  classes = classifieur.classes_ if nom.startswith("Emb") else np.array(intentions)
  confiance = proba.max(axis=1)
  justes = classes[proba.argmax(axis=1)] == ...  # les vraies intentions du test
  print(f"{nom:25} confiance {confiance.mean():.2f}  justes {justes.mean():.3f}")
  part_justes, confiance_moyenne = calibration_curve(
    ..., ..., n_bins=10, strategy="quantile"
  )
  plt.plot(confiance_moyenne, part_justes, "o-", label=nom)
plt.plot([0, 1], [0, 1], "--", color="gray", label="Calibration parfaite")
plt.xlabel("Probabilité annoncée")
plt.ylabel("Part de prédictions justes")
plt.legend()
plt.show()
```

#### Solution

In [ ]:
import matplotlib.pyplot as plt
from scipy.special import softmax
from sklearn.calibration import calibration_curve

probabilites = {
  "Embeddings + régression": classifieur.predict_proba(E_test),
  "Encodeur affiné": softmax(logits, axis=1),
}
for nom, proba in probabilites.items():
  classes = classifieur.classes_ if nom.startswith("Emb") else np.array(intentions)
  confiance = proba.max(axis=1)
  justes = classes[proba.argmax(axis=1)] == test.label.to_numpy()
  print(f"{nom:25} confiance {confiance.mean():.2f}  justes {justes.mean():.3f}")
  part_justes, confiance_moyenne = calibration_curve(
    justes, confiance, n_bins=10, strategy="quantile"
  )
  plt.plot(confiance_moyenne, part_justes, "o-", label=nom)
plt.plot([0, 1], [0, 1], "--", color="gray", label="Calibration parfaite")
plt.xlabel("Probabilité annoncée")
plt.ylabel("Part de prédictions justes")
plt.legend()
plt.show()

# Avec nos vecteurs et nos logits publiés (sans GPU) ; sur GPU, des chiffres voisins.
# La régression sur embeddings est la mieux calibrée : confiance moyenne 0,82 pour
# 85,5 % de justes, et sa courbe suit la diagonale, un peu au-dessus (elle
# se sous-estime légèrement). L'encodeur affiné, plus juste, est sur-confiant :
# confiance moyenne 0,95 pour 87,5 % de justes, et sa courbe passe sous la diagonale.
# Les trois quarts de ses prédictions annoncent plus de 0,99, et 96 % seulement de
# celles-là sont justes : avant de décider sur sa probabilité (un seuil, un renvoi à
# un humain), il faut le recalibrer sur un jeu de validation.